In [1]:
import serial
import time

In [2]:
import serial.tools.list_ports

ports = list(serial.tools.list_ports.comports())
for p in ports:
    print(p.device, p.description)

COM3 Silicon Labs CP210x USB to UART Bridge (COM3)


In [3]:
obj = serial.Serial('COM3', 9600, timeout=1)

print(obj)

obj.close()

Serial<id=0x264bb4cc6a0, open=True>(port='COM3', baudrate=9600, bytesize=8, parity='N', stopbits=1, timeout=1, xonxoff=False, rtscts=False, dsrdtr=False)


In [4]:
ser = serial.Serial(
    port="COM3",          # /dev/ttyUSB0 on Raspberry Pi
    baudrate=9600,
    bytesize=8,
    parity='N',
    stopbits=1,
    timeout=1
)

In [5]:
def send_frame(frame: str):
    time.sleep(0.01)              # ≥ 3.5 char silence
    ser.write(frame.encode('ascii'))
    ser.flush()

In [6]:
def read_frame():
    buf = ""
    while True:
        ch = ser.read(1).decode('ascii', errors='ignore')
        if not ch:
            return None
        buf += ch
        if ch == '>':
            return buf

In [21]:
def set_voltage(v):
    if v < 0 or v > 31:
        raise ValueError("Voltage must be between 0 and 31 V")
    value_split = str(v).split('.')
    if len(value_split[0]) > 3 or len(value_split[1]) > 3:
        raise ValueError("Voltage must have at most 3 digits before and after the decimal point")
    value_str = f"{int(float(v) * 1000):06d}"
    frame = f"<01{value_str}000>"
    send_frame(frame)
    print("Sent frame:", frame)
    return read_frame()

def read_voltage():
    send_frame("<02000000000>")
    resp = read_frame()
    value = int(resp[3:10]) / 10000
    print("Received frame:", resp, "Value:", value)
    return value

def set_current(i):
    if i < 0 or i > 10:
        raise ValueError("Current must be between 0 and 10 A")
    value_split = str(i).split('.')
    if len(value_split[0]) > 3 or len(value_split[1]) > 3:
        raise ValueError("Current must have at most 3 digits before and after the decimal point")
    value_str = f"{int(float(i) * 1000):06d}"
    frame = f"<03{value_str}000>"
    send_frame(frame)
    print("Sent frame:", frame)
    return read_frame()

def read_current():
    send_frame("<04000000000>")
    resp = read_frame()
    value = int(resp[3:10]) / 10000
    print("Received frame:", resp, "Value:", value)
    return value

In [22]:
def power_on():
    send_frame("<07000000000>")
    return read_frame()

def power_off():
    send_frame("<08000000000>")
    return read_frame()

def pc_connect():
    send_frame("<09100000000>")
    return read_frame()

def pc_disconnect():
    send_frame("<09200000000>")
    return read_frame()

In [23]:
pc_connect()
read_frame()

In [24]:
set_voltage(11.136)
set_current(4.0)

Sent frame: <01011136000>
Sent frame: <03004000000>


'<13OK0049000>'

In [25]:
power_on()
print(read_voltage(), read_current())

Received frame: <12011136000> Value: 11.136
Received frame: <14001842000> Value: 1.842
11.136 1.842


In [26]:
power_off()
pc_disconnect()

'<19OK0042000>'